# v24.3 Worst-case A — Freeze MC + default YNU
No grid. No model. Checks safe lower-bound without over-tuning.

In [ ]:

# ==================================================================
# COMMON -- imports, paths, helpers
# ==================================================================
import os, re, json, glob, math, itertools, statistics
from pathlib import Path
from collections import Counter, defaultdict

OUT_DIR = Path('/kaggle/working/v24_3_freeze_mc_tune_ynu')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MC_LABELS = {'A','B','C','D'}
YNU_LABELS = {'Yes','No','Unknown'}
ALL_LABELS = ['A','B','C','D','Yes','No','Unknown']

# User can override these manually if resolver misses.
CAND_PATH_OVERRIDE = ''
MC_DEBUG_PATH_OVERRIDE = ''
V24_3_BEST_PARAMS_OVERRIDE = ''


def find_first(patterns, required=True, desc='file'):
    hits=[]
    for pat in patterns:
        hits.extend(glob.glob(pat, recursive=True))
    # prefer exact useful names, newest-ish lexical after working/input
    hits = sorted(set(hits), key=lambda p: (('/kaggle/working' not in p), len(p), p))
    if hits:
        print(f'Found {desc}:', hits[0])
        return Path(hits[0])
    if required:
        raise FileNotFoundError(f'Could not find {desc}. Tried patterns:\n' + '\n'.join(patterns))
    print(f'Optional {desc} not found')
    return None

CAND_PATH = Path(CAND_PATH_OVERRIDE) if CAND_PATH_OVERRIDE else find_first([
    '/kaggle/input/**/v23_val_candidates.json',
    '/kaggle/working/**/v23_val_candidates.json',
], desc='v23_val_candidates.json')

MC_DEBUG_PATH = Path(MC_DEBUG_PATH_OVERRIDE) if MC_DEBUG_PATH_OVERRIDE else find_first([
    '/kaggle/input/**/v24_1_optionwise_mc_debug*.json',
    '/kaggle/working/**/v24_1_optionwise_mc_debug*.json',
], required=False, desc='v24.1 optionwise MC debug json')

with open(CAND_PATH, 'r', encoding='utf-8') as f:
    results = json.load(f)
print('Loaded candidates:', len(results))
print('First result keys:', sorted(results[0].keys()))
print('First candidate keys:', sorted(results[0].get('candidates', [{}])[0].keys()))

mc_debug = None
if MC_DEBUG_PATH and Path(MC_DEBUG_PATH).exists():
    with open(MC_DEBUG_PATH, 'r', encoding='utf-8') as f:
        mc_debug = json.load(f)
    print('Loaded MC debug records:', len(mc_debug))
else:
    print('No MC debug; MC will fallback to candidate majority/oracle-style candidate rerank depending notebook mode.')


def norm_answer(x):
    if x is None:
        return 'UNPARSEABLE'
    s = str(x).strip()
    s2 = s.lower()
    if s.upper() in MC_LABELS:
        return s.upper()
    if s2 in {'yes','true','y'}:
        return 'Yes'
    if s2 in {'no','false','n'}:
        return 'No'
    if s2 in {'unknown','unk','cannot determine','not enough information'}:
        return 'Unknown'
    m = re.search(r'\b(A|B|C|D|Yes|No|Unknown)\b', s, flags=re.I)
    if m:
        return norm_answer(m.group(1))
    return 'UNPARSEABLE'


def get_gold(r):
    for k in ['gold','answer','label','gold_answer','target']:
        if k in r:
            return norm_answer(r.get(k))
    return 'UNPARSEABLE'


def cand_ans(c):
    for k in ['answer','pred','final_answer','parsed_answer']:
        if k in c:
            return norm_answer(c.get(k))
    txt = str(c.get('text') or c.get('raw') or c.get('output') or '')
    m = re.search(r'Final\s*Answer\s*[:\-]?\s*(A|B|C|D|Yes|No|Unknown)', txt, flags=re.I)
    return norm_answer(m.group(1)) if m else 'UNPARSEABLE'


def cand_text(c):
    return str(c.get('text') or c.get('raw') or c.get('output') or c.get('explanation') or '')


def cand_supp(c):
    x = c.get('supporting_premises') or c.get('support') or c.get('cited_premises') or []
    if isinstance(x, str):
        nums = re.findall(r'\d+', x)
        return [int(n) for n in nums]
    if isinstance(x, (list, tuple)):
        out=[]
        for v in x:
            try: out.append(int(v))
            except Exception: pass
        return out
    return []


def is_mc_val_record(r):
    # For validation only: strict gold-based split to avoid prior over-detection.
    return get_gold(r) in MC_LABELS


def majority_pred(r, allowed=None):
    allowed = allowed or set(ALL_LABELS)
    votes=[]
    for c in r.get('candidates', []):
        a = cand_ans(c)
        if a in allowed:
            votes.append(a)
    if not votes:
        return 'Unknown'
    cnt=Counter(votes)
    # stable tie: by count then label order
    order = ['A','B','C','D','Yes','No','Unknown']
    return sorted(cnt.keys(), key=lambda a: (-cnt[a], order.index(a) if a in order else 999))[0]


def metric(y_true, y_pred, name='metric'):
    n=len(y_true)
    per={}
    for lab in ALL_LABELS:
        tp=sum(1 for g,p in zip(y_true,y_pred) if g==lab and p==lab)
        fp=sum(1 for g,p in zip(y_true,y_pred) if g!=lab and p==lab)
        fn=sum(1 for g,p in zip(y_true,y_pred) if g==lab and p!=lab)
        prec=tp/(tp+fp) if tp+fp else 0.0
        rec=tp/(tp+fn) if tp+fn else 0.0
        f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
        support=sum(1 for g in y_true if g==lab)
        pred_count=sum(1 for p in y_pred if p==lab)
        per[lab]={'precision':prec,'recall':rec,'f1':f1,'support':support,'pred_count':pred_count,'tp':tp,'fp':fp,'fn':fn}
    acc=sum(g==p for g,p in zip(y_true,y_pred))/n if n else 0.0
    macro=sum(per[l]['f1'] for l in ALL_LABELS)/len(ALL_LABELS)
    weighted=sum(per[l]['f1']*per[l]['support'] for l in ALL_LABELS)/max(1,n)
    mc_labels=['A','B','C','D']; ynu_labels=['Yes','No','Unknown']
    mc_macro=sum(per[l]['f1'] for l in mc_labels)/4
    ynu_macro=sum(per[l]['f1'] for l in ynu_labels)/3
    return {'name':name,'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro_f1':mc_macro,'ynu_macro_f1':ynu_macro,'per_label':per}


def print_metric(m):
    print('='*100)
    print(f"{m['name']} | n={m['n']} acc={m['acc']:.3f} macro_f1={m['macro_f1']:.3f} weighted={m['weighted_f1']:.3f} MC={m['mc_macro_f1']:.3f} YNU={m['ynu_macro_f1']:.3f}")
    print('-'*100)
    print(f"{'Label':<10}{'P':>8}{'R':>8}{'F1':>8}{'Gold':>8}{'Pred':>8}")
    for lab in ALL_LABELS:
        d=m['per_label'][lab]
        print(f"{lab:<10}{d['precision']:>8.3f}{d['recall']:>8.3f}{d['f1']:>8.3f}{d['support']:>8}{d['pred_count']:>8}")


def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x, (np.integer,)): return int(x)
        if isinstance(x, (np.floating,)): return float(x)
        if isinstance(x, (np.bool_,)): return bool(x)
    except Exception:
        pass
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

golds=[get_gold(r) for r in results]
print('Gold dist:', Counter(golds))
print('Strict MC count:', sum(g in MC_LABELS for g in golds), 'YNU count:', sum(g in YNU_LABELS for g in golds))


In [ ]:

# ==================================================================
# CORE -- frozen MC from v24.1 debug + tunable YNU reranker
# ==================================================================

def optionwise_mc_pred_from_debug(idx, r, tie_mode='majority_then_order'):
    """Use saved v24.1 option-wise support decisions. No model generation here."""
    if mc_debug is None or str(idx) not in mc_debug:
        return majority_pred(r, allowed=MC_LABELS)
    supp = mc_debug[str(idx)].get('support', {}) or {}
    true_opts = [o for o in ['A','B','C','D'] if bool(supp.get(o, False))]
    if not true_opts:
        return majority_pred(r, allowed=MC_LABELS)
    if len(true_opts) == 1:
        return true_opts[0]
    # tie-break using candidate vote among supported options, then support text heuristic, then label order
    cnt = Counter(cand_ans(c) for c in r.get('candidates', []) if cand_ans(c) in true_opts)
    if cnt:
        best_count=max(cnt.values())
        best=[o for o in true_opts if cnt[o]==best_count]
        if len(best)==1:
            return best[0]
        true_opts=best
    raw = mc_debug[str(idx)].get('raw', {}) or {}
    def raw_score(o):
        txt=str(raw.get(o,'')).lower()
        s=0.0
        for phrase in ['directly support','direct restatement','explicitly stated','is entailed','therefore']:
            if phrase in txt: s += 0.15
        for phrase in ['not entailed','cannot infer','not supported','contradict']:
            if phrase in txt: s -= 0.25
        return s
    scores={o: raw_score(o) for o in true_opts}
    best=max(true_opts, key=lambda o: (scores[o], -['A','B','C','D'].index(o)))
    return best


def candidate_vote_share(r, ans):
    vals=[cand_ans(c) for c in r.get('candidates', [])]
    vals=[v for v in vals if v in ALL_LABELS]
    if not vals: return 0.0
    return vals.count(ans)/len(vals)


def ynu_score_candidate(c, r, params):
    ans = cand_ans(c)
    if ans not in YNU_LABELS:
        return -999
    txt = cand_text(c).lower()
    supp = cand_supp(c)
    vote = candidate_vote_share(r, ans)
    citation = 1.0 if len(supp) > 0 else 0.0
    short = 1.0 if 1 <= len(supp) <= 5 else 0.0
    long = 1.0 if len(supp) > 8 else 0.0
    contra = 1.0 if any(p in txt for p in ['contradict', 'not support', 'not entailed', 'cannot infer', 'insufficient']) else 0.0
    score = 0.0
    score += params.get('w_vote', 1.0) * vote
    score += params.get('w_cite', 0.0) * citation
    score += params.get('w_short_supp', 0.0) * short
    score -= params.get('p_long_supp', 0.0) * long
    score -= params.get('p_contra_text', 0.0) * contra
    if ans == 'Yes': score += params.get('yes_boost', 0.0)
    if ans == 'No': score -= params.get('no_penalty', 0.0)
    if ans == 'Unknown': score -= params.get('unk_penalty', 0.0)
    return score


def ynu_rerank_pred(r, params):
    best_a='Unknown'; best_s=-1e9
    for c in r.get('candidates', []):
        a=cand_ans(c)
        if a not in YNU_LABELS:
            continue
        s=ynu_score_candidate(c, r, params)
        if s > best_s:
            best_s=s; best_a=a
    return best_a


def predict_all(params, freeze_mc=True):
    preds=[]
    for idx,r in enumerate(results):
        g=get_gold(r)
        if g in MC_LABELS and freeze_mc:
            preds.append(optionwise_mc_pred_from_debug(idx, r))
        elif g in MC_LABELS:
            preds.append(majority_pred(r, allowed=MC_LABELS))
        else:
            preds.append(ynu_rerank_pred(r, params))
    return preds

base_params = dict(w_vote=1.0, w_cite=0.3, w_short_supp=0.15, yes_boost=0.3, no_penalty=0.15, unk_penalty=0.35, p_long_supp=0.2, p_contra_text=0.4)
first_preds=[next((cand_ans(c) for c in r.get('candidates', []) if cand_ans(c) in ALL_LABELS), 'Unknown') for r in results]
maj_preds=[majority_pred(r) for r in results]
base_preds=predict_all(base_params, freeze_mc=True)
for m in [metric(golds, first_preds, 'first'), metric(golds, maj_preds, 'majority'), metric(golds, base_preds, 'v24_3_base_freeze_mc_default_ynu')]:
    print_metric(m)


In [ ]:

# ==================================================================
# WORST-CASE A -- no YNU grid: freeze MC + safest current YNU rule
# ==================================================================
# Use conservative/default params only. This checks if v24.3 is useful without overfitting a grid.
params = dict(w_vote=1.0, w_cite=0.3, w_short_supp=0.15, yes_boost=0.3, no_penalty=0.15, unk_penalty=0.35, p_long_supp=0.2, p_contra_text=0.4)
preds = predict_all(params, freeze_mc=True)
m = metric(golds, preds, 'v24_3_worstcase_A_freeze_mc_default_ynu')
print_metric(m)
summary={'params':params,'metric':m}
with open(OUT_DIR/'v24_3_worstcase_A_summary.json','w',encoding='utf-8') as f:
    json.dump(to_jsonable(summary), f, ensure_ascii=False, indent=2)
print('Saved:', OUT_DIR/'v24_3_worstcase_A_summary.json')
